# End-to-End Financial Analysis and Forecasting

## Project Purpose

This notebook presents a full financial analysis and forecasting workflow using simulated company data. 

The analysis answers practical business questions:

1. What is driving revenue?
2. What is driving profit?
3. Which products and regions are most profitable?
4. Where is the company beating or missing budget?
5. Are discounts helping or hurting profitability?
6. Which customers are most valuable?
7. How efficiently does the company convert sales into cash?
8. Can monthly revenue be forecasted?
9. What actions should management take?

The project uses invoice-level data, monthly budget data, financial statement data, and macroeconomic indicators.

### 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.holtwinters import ExponentialSmoothing

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# Project folders
DATA_DIR = Path("../data")
FIGURES_DIR = Path("../figures")
OUTPUTS_DIR = Path("../outputs")

FIGURES_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

### 2. Load the Data

This project uses four related datasets.

1. `simulated_financial_transactions.csv`: invoice-level data
2. `monthly_budget_targets.csv`: budgeted revenue and costs
3. `monthly_financial_statements.csv`: monthly accounting and cash flow metrics
4. `macro_indicators_monthly.csv`: external variables for forecasting

In [ ]:
transactions = pd.read_csv(DATA_DIR / "simulated_financial_transactions.csv")
budget = pd.read_csv(DATA_DIR / "monthly_budget_targets.csv")
financials = pd.read_csv(DATA_DIR / "monthly_financial_statements.csv")
macro = pd.read_csv(DATA_DIR / "macro_indicators_monthly.csv")

print("Transactions shape:", transactions.shape)
print("Budget shape:", budget.shape)
print("Financial statements shape:", financials.shape)
print("Macro indicators shape:", macro.shape)

In [ ]:
transactions.head()

### 3. Initial Data Inspection

Before doing any financial analysis, a professional analyst should inspect the data for:

- Missing values
- Duplicate rows
- Incorrect data types
- Inconsistent categorical values
- Outliers
- Date problems

In [ ]:
def inspect_dataframe(df, name):
    print(f"\n=================={name.upper()} =====================")
    print("Shape:", df.shape)
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isna().sum().sort_values(ascending=False).head())
    print("\nDuplicate rows:", df.duplicated().sum())

inspect_dataframe(transactions, "transactions")
inspect_dataframe(budget, "budget")
inspect_dataframe(financials, "financials")
inspect_dataframe(macro, "macro")

### 4. Data Cleaning

Cleaning steps:

1. Convert date columns to datetime.
2. Remove duplicate invoice rows.
3. Standardize text columns.
4. Fill missing financial rates with appropriate values.
5. Recalculate gross margin where needed.
6. Create monthly date variables.

In [ ]:
# Convert dates
transactions["invoice_date"] = pd.to_datetime(transactions["invoice_date"])
budget["month"] = pd.to_datetime(budget["month"])
financials["month"] = pd.to_datetime(financials["month"])
macro["month"] = pd.to_datetime(macro["month"])

In [ ]:
# Remove duplicates in transactions
before = len(transactions)
transactions = transactions.drop_duplicates()
after = len(transactions)

print(f"Duplicate rows removed: {before - after}")

In [ ]:
# Clean the text columns (not needed)
text_cols = transactions.select_dtypes(include='object').columns

for col in text_cols:
    transactions[col] = transactions[col].str.strip().str.title()

In [ ]:
# Fill missing values with their median
transactions["discount_rate"] = transactions["discount_rate"].fillna(
    transactions["discount_rate"].median())

# Fill missing days to collect by payment terms group with median
transactions["days_to_collect"] = transactions.groupby("payment_terms")["days_to_collect"].transform(
    lambda x: x.fillna(x.median())) # For every group, calculates the median to fill NA

transactions["gross_margin_pct"] = transactions["gross_margin_pct"].fillna(
    transactions["gross_profit"] / transactions["net_revenue"]) # recalculate gross margin where missing

In [ ]:
# create time variables
transactions["month"] = transactions["invoice_date"].dt.to_period("M").dt.to_timestamp()
transactions["year"] = transactions["invoice_date"].dt.year
transactions["quarter"] = transactions["invoice_date"].dt.quarter
transactions["month_name"] = transactions["invoice_date"].dt.month_name()

In [ ]:
# Save Cleaned data
transactions.to_csv(OUTPUTS_DIR / "cleaned_financial_transactions.csv", index = False)
transactions.head()

### KPI Summary

Key metrics include:

- Revenue
- COGS
- Gross profit
- Gross margin
- EBITDA
- EBITDA margin
- Net income
- Net margin
- Free cash flow
- Return rate
- Days to collect

In [ ]:
financials.head()

In [ ]:
# current year KPI
latest_year = financials["month"].dt.year.max()

fy_summary = financials[financials["month"].dt.year == latest_year].agg({
    "revenue": "sum",
    "cogs": "sum",
    "gross_profit": "sum",
    "ebitda": "sum",
    "net_income": "sum",
    "free_cash_flow": "sum",
    "return_rate": "mean",
    "avg_days_to_collect": "mean"
})

executive_kpis = pd.DataFrame({
    "Metric": [
        "Fiscal Year",
        "Revenue",
        "COGS",
        "Gross Profit",
        "Gross Margin",
        "EBITDA",
        "EBITDA Margin",
        "Net Income",
        "Net Margin",
        "Free Cash Flow",
        "Return Rate",
        "Average Days to Collect"
    ],
    "Value": [
        latest_year,
        fy_summary["revenue"],
        fy_summary["cogs"],
        fy_summary["gross_profit"],
        fy_summary["gross_profit"] / fy_summary["revenue"],
        fy_summary["ebitda"],
        fy_summary["ebitda"] / fy_summary["revenue"],
        fy_summary["net_income"],
        fy_summary["net_income"] / fy_summary["revenue"],
        fy_summary["free_cash_flow"],
        fy_summary["return_rate"],
        fy_summary["avg_days_to_collect"]
    ]
})

executive_kpis.to_csv(OUTPUTS_DIR / "executive_kpi_summary.csv", index=False)
executive_kpis

### Revenue Analysis

We analyze revenue by:

- Product line
- Region
- Sales channel
- Customer segment
- Industry

The goal is to understand growth drivers and resource allocation.

In [ ]:
transactions.head()

In [ ]:
revenue_by_product = (
    transactions
    .groupby("product_line", as_index=False)
    .agg(
        revenue=("net_revenue", "sum"),
        gross_profit=("gross_profit", "sum"),
        units=("units", "sum"),
        avg_discount=("discount_rate", "mean"),
        return_rate=("is_returned", "mean"),
        customers=("customer_id", "nunique")
    )
)

revenue_by_product["gross_margin"] = revenue_by_product["gross_profit"] / revenue_by_product["revenue"]
revenue_by_product["revenue_share"] = revenue_by_product["revenue"] / revenue_by_product["revenue"].sum()
revenue_by_product = revenue_by_product.sort_values("revenue", ascending=False)

revenue_by_product.to_csv(OUTPUTS_DIR / "revenue_by_product_line.csv", index=False)
revenue_by_product

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(
    data=revenue_by_product,
    x="product_line",
    y="revenue"
)
plt.title("Revenue by Product Line")
plt.xlabel("Product Line")
plt.ylabel("Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "revenue_by_product_line.png", dpi=300)
plt.show()

In [ ]:
revenue_by_region = (
    transactions
    .groupby("region", as_index=False)
    .agg(
        revenue=("net_revenue", "sum"),
        gross_profit=("gross_profit", "sum"),
        customers=("customer_id", "nunique"),
        avg_days_to_collect=("days_to_collect", "mean"),
        return_rate=("is_returned", "mean")
    )
)

revenue_by_region["gross_margin"] = revenue_by_region["gross_profit"] / revenue_by_region["revenue"]
revenue_by_region = revenue_by_region.sort_values("revenue", ascending=False)

revenue_by_region.to_csv(OUTPUTS_DIR / "revenue_by_region.csv", index=False)
revenue_by_region

In [ ]:
transactions.head()

In [ ]:
fig = px.treemap(
    transactions,
    path=["region", "product_line", "sales_channel"],
    values="net_revenue",
    color="gross_profit",
    title="Revenue Composition by Region, Product Line, and Sales Channel"
)

fig.show()

# Save interactive and static outputs
fig.write_html(FIGURES_DIR / "revenue_composition_treemap.html")
fig.write_image(FIGURES_DIR / "revenue_composition_treemap.png", width=1100, height=700)

### Profitability Analysis
Revenue growth is not enough. We need to know whether revenue is profitable.

Here, we compare revenue and gross margin by product line. A product may have high sales but weak margin, which may require pricing changes, supplier negotiations, or discount controls.

In [ ]:
profitability = revenue_by_product.copy()

portfolio_gross_margin = transactions["gross_profit"].sum() / transactions["net_revenue"].sum()

profitability["margin_gap_vs_portfolio"] = profitability["gross_margin"] - portfolio_gross_margin
profitability["high_revenue_weak_margin"] = (
    (profitability["revenue_share"] > profitability["revenue_share"].median()) &
    (profitability["gross_margin"] < portfolio_gross_margin)
)

profitability.to_csv(OUTPUTS_DIR / "profitability_by_product_line.csv", index=False)
profitability

In [ ]:
revenue_by_product.head()